In [1]:
from dotenv import load_dotenv
from langchain_groq import ChatGroq

from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr

c:\Users\musta\OneDrive\Desktop\llm-lab\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm_model = "qwen/qwen3.8-27b"
DB_NAME = "vector_db"
load_dotenv(override=True)

True

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5019.97it/s]


In [4]:
retriever = vectorstore.as_retriever()
llm = ChatGroq(temperature=0, model_name=llm_model)

In [5]:
retriever.invoke("Who is Avery?")

[Document(id='6dfa1d54-fc3c-4f3b-ae5a-5caeb52612a6', metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'}, page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- **Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social responsibility image.  \n\nAvery Lancaster has demonstrated resilience and a

In [6]:
llm.invoke("Who is Avery?")

AIMessage(content='The name **Avery** is a unisex name that can refer to several notable individuals, depending on the context. Here are some of the most prominent people named Avery:\n\n### 1. **Avery Brundage (1902–1987)**\n   - An American businessman and philanthropist.\n   - Best known as the long-time president of the **American Library Association (ALA)** and for his controversial role in the development of **MARC (Machine-Readable Cataloging)**, which revolutionized library cataloging.\n\n### 2. **Avery Johnson (1949–2023)**\n   - A former professional basketball player and coach.\n   - Played in the NBA for the Houston Rockets and others.\n   - Served as head coach of the **Houston Rockets** and **San Antonio Spurs**, winning an NBA championship with the Spurs in 2003.\n\n### 3. **Avery Tipler (1923–2013)**\n   - An American theoretical physicist.\n   - Known for his work in **general relativity**, **cosmology**, and **quantum field theory**.\n   - Co-authored the influential 

In [7]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [8]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [9]:
answer_question("Who is Averi Lancaster?", [])

'Based on the context provided, **Avery Lancaster** (note: the context spells it "Avery," while your question uses "Averi") is the **Co-Founder and Chief Executive Officer (CEO)** of Insurellm.\n\nHere are the key details about her:\n\n*   **Role:** She co-founded Insurellm in 2015 and has led the company to become a leading Insurance Tech provider.\n*   **Location:** San Francisco, California.\n*   **Background:** Before founding Insurellm, she worked as a Senior Product Manager at Innovate Insurance Solutions (2013–2015), where she developed insurance products for the tech sector.\n*   **Leadership Style:** She is known for her innovative leadership strategies and risk management expertise.\n*   **Salary:** Her current salary is listed as $225,000.\n\n*(Note: The context also contains a section labeled "Avery Lancaster" that includes details about a "Senior Data Engineer" named Maxine, but this appears to be a formatting error in the source text, as the name and role do not match. Th

In [10]:
gr.ChatInterface(answer_question).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
